# 04 — Temperatura do ar, umidade, sensação térmica e índices de conforto no horário de aula
O satélite mede **superfície**, uma vez por dia (~10h). Para sensação térmica precisamos de **ar + umidade, hora a hora**.

Abordagem:
1. Série horária de referência (T_ar, UR) das estações INMET da capital (Mirante de Santana A701, Interlagos A771).
2. Diferença local de cada escola: `ΔT_escola = β · (ANOM_escola − ANOM_estação)` — a anomalia de superfície
   do notebook 03 vira anomalia de ar via coeficiente β (padrão 0,3; calibrar com estações CGE se disponíveis).
3. Umidade ajustada mantendo a pressão de vapor constante (ar mais quente → UR menor).
4. Índices só em **horas letivas** (dias úteis, meses letivos, 7h–19h).

⚠️ Como ΔT é constante por escola, a **ordem** entre escolas vem da anomalia de superfície; os índices traduzem isso
em °C, horas e graus-hora interpretáveis. Calibrar β com mais estações (CGE) melhora a magnitude, não a ordem.

In [ ]:
!pip -q install geopandas rasterio

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import requests, zipfile, io, glob, re, numpy as np, pandas as pd, geopandas as gpd
from indices import heat_index, thom_di

ANOS = [2023, 2024, 2025]
ESTACOES = {'A701': 'SAO PAULO - MIRANTE', 'A771': 'SAO PAULO - INTERLAGOS'}
BETA = 0.3                     # °C de ar por °C de anomalia de superfície (literatura: 0,2–0,5)
HORAS_AULA = range(7, 19)      # 7h00–18h59
MESES_LETIVOS = [2, 3, 4, 5, 6, 8, 9, 10, 11, 12]
LIMIAR_GH = 26.0               # °C para graus-hora

In [ ]:
os.makedirs(f'{RAW}/inmet', exist_ok=True)
def baixa_inmet(ano):
    url = f'https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip'
    z = zipfile.ZipFile(io.BytesIO(requests.get(url, timeout=600).content))
    for n in z.namelist():
        if any(cod in n.upper() for cod in ESTACOES):
            alvo = f"{RAW}/inmet/{os.path.basename(n)}"
            open(alvo, 'wb').write(z.read(n)); print(alvo)
for a in ANOS:
    if not glob.glob(f'{RAW}/inmet/*{a}*'):
        baixa_inmet(a)

In [ ]:
def le_inmet(p):
    meta = pd.read_csv(p, sep=';', encoding='latin-1', nrows=7, header=None, index_col=0)[1]
    meta.index = meta.index.str.strip(': ').str.upper()
    lat = float(str(meta.filter(like='LATITUDE').iloc[0]).replace(',', '.'))
    lon = float(str(meta.filter(like='LONGITUDE').iloc[0]).replace(',', '.'))
    d = pd.read_csv(p, sep=';', encoding='latin-1', skiprows=8, decimal=',', dtype=str)
    c_data = [c for c in d.columns if c.upper().startswith('DATA')][0]
    c_hora = [c for c in d.columns if c.upper().startswith('HORA')][0]
    c_t = [c for c in d.columns if 'BULBO SECO' in c.upper()][0]
    c_ur = [c for c in d.columns if 'UMIDADE RELATIVA DO AR, HORARIA' in c.upper()][0]
    h = d[c_hora].str.extract(r'(\d{1,2})')[0].str.zfill(2)
    ts = pd.to_datetime(d[c_data].str.replace('/', '-') + ' ' + h + ':00', utc=True).dt.tz_convert('America/Sao_Paulo')
    out = pd.DataFrame({'datahora': ts,
                        'T': pd.to_numeric(d[c_t].str.replace(',', '.'), errors='coerce'),
                        'UR': pd.to_numeric(d[c_ur].str.replace(',', '.'), errors='coerce')})
    out.loc[out['T'] < -50, 'T'] = np.nan; out.loc[out['UR'] < 0, 'UR'] = np.nan
    cod = re.search(r'A\d{3}', os.path.basename(p).upper()).group(0)
    return out.assign(estacao=cod, lat=lat, lon=lon)

met = pd.concat([le_inmet(p) for p in sorted(glob.glob(f'{RAW}/inmet/*.CSV') + glob.glob(f'{RAW}/inmet/*.csv'))])
print(met.groupby('estacao')[['T', 'UR']].describe().round(1))

In [ ]:
# série de referência = média horária das estações; mantém só horas letivas
ref = met.groupby('datahora')[['T', 'UR']].mean().dropna()
ref = ref[(ref.index.dayofweek < 5) & ref.index.month.isin(MESES_LETIVOS) & ref.index.hour.isin(HORAS_AULA)]
print(len(ref), 'horas letivas na série')

# anomalia de superfície nas estações (mesmo buffer das escolas)
from exactextract import exact_extract
est = met.groupby('estacao')[['lat', 'lon']].first().reset_index()
est_g = gpd.GeoDataFrame(est, geometry=gpd.points_from_xy(est.lon, est.lat), crs=4326).to_crs(CRS)
est_g['geometry'] = est_g.buffer(BUFFER_REF)
TIF = (glob.glob(f'{INT}/lst_composto.vrt') or sorted(glob.glob(f'{RAW}/lst/lst_composto*.tif')))[0]
import rasterio
ORDEM_GEE = ['LST_min','LST_mean','LST_median','LST_p90','LST_max','LST_stdDev','ANOM_mean','ANOM_p90','NDVI_median']
with rasterio.open(TIF) as r: nomes = list(r.descriptions)
if not all(nomes): nomes = ORDEM_GEE
i_anom = nomes.index('ANOM_mean') + 1
anom_est = exact_extract(TIF, est_g, 'mean', include_cols=['estacao'], output='pandas')
anom_ref = float(anom_est[f'band_{i_anom}_mean' if f'band_{i_anom}_mean' in anom_est else 'ANOM_mean_mean'].mean())
print('anomalia média nas estações:', round(anom_ref, 2))

### (Opcional) calibrar β com a rede CGE
Se você obtiver dados horários das estações do CGE (pedido via LAI/SIURB), salve em
`data/raw/cge/cge_horario.csv` com colunas `estacao, lat, lon, datahora, T, UR`. A célula abaixo estima β por
regressão de (T_média_estação − T_média_cidade) contra a anomalia de superfície no entorno da estação.

In [ ]:
p = f'{RAW}/cge/cge_horario.csv'
if os.path.exists(p):
    cge = pd.read_csv(p, parse_dates=['datahora'])
    m = cge.groupby('estacao').agg(T=('T', 'mean'), lat=('lat', 'first'), lon=('lon', 'first')).reset_index()
    m['dT'] = m['T'] - m['T'].mean()
    g = gpd.GeoDataFrame(m, geometry=gpd.points_from_xy(m.lon, m.lat), crs=4326).to_crs(CRS); g['geometry'] = g.buffer(BUFFER_REF)
    a = exact_extract(TIF, g, 'mean', include_cols=['estacao'], output='pandas')
    m['anom'] = a.iloc[:, i_anom].values
    BETA = float(np.polyfit(m['anom'] - m['anom'].mean(), m['dT'], 1)[0])
    print('β calibrado =', round(BETA, 3), '| n estações =', len(m))
else:
    print('Sem dados CGE — usando β =', BETA)

In [ ]:
temp = pd.read_parquet(f'{PRO}/temperatura_escolas.parquet')
dT = (BETA * (temp['ANOM_mean'] - anom_ref)).to_numpy(np.float32)          # (N,)
T = ref['T'].to_numpy(np.float32)[None, :] + dT[:, None]                    # (N, H)

def es(t): return 6.112 * np.exp(17.62 * t / (243.12 + t))                # hPa (Magnus)
e = (ref['UR'].to_numpy(np.float32) / 100) * es(ref['T'].to_numpy(np.float32))
UR = np.clip(100 * e[None, :] / es(T), 1, 100)

HI = heat_index(T, UR).astype(np.float32)
DI = thom_di(T, UR).astype(np.float32)
anos = len(set(ref.index.year))

ind = pd.DataFrame({
    'cod_esc': temp['cod_esc'],
    'dT_ar_estimado': dT,
    'Tar_min_aula': T.min(1), 'Tar_media_aula': T.mean(1), 'Tar_p95_aula': np.percentile(T, 95, axis=1),
    'Tar_max_aula': T.max(1), 'Tar_dp_aula': T.std(1),
    'HI_media_aula': HI.mean(1), 'HI_max_aula': HI.max(1),
    'DI_media_aula': DI.mean(1), 'DI_max_aula': DI.max(1),
    'GH26_ano': np.clip(T - LIMIAR_GH, 0, None).sum(1) / anos,
    'h_HI_ge32_ano': (HI >= 32).sum(1) / anos,
    'h_DI_ge24_ano': (DI >= 24).sum(1) / anos,
    'h_DI_ge27_ano': (DI >= 27).sum(1) / anos,
})
ind.to_parquet(f'{PRO}/indices_escolas.parquet')
ind.describe().T.round(2)